# 第3回B 演習: Forth インタプリタの実装

スライド: [第3回B](../03b-forth/)

スタック言語 Forth のインタプリタを、算術 → 組み込みワード → 定義 → REPL の順に足していく。

```
3 4 + .          →  7
: SQUARE DUP * ;
5 SQUARE .       →  25
```

## 進め方

- `TODO` を埋めて、直下の確認セルを実行する
- 確認セル (`# [CHECK]`) は自分用。消しても教員の採点は別途走るが、残すこと

| 課題 | 配点 |
|---|---|
| 課題1 算術だけのインタプリタ | 20 |
| 課題2 スタック操作ワード | 20 |
| 課題3 ユーザ定義ワード | 25 |
| 課題4 REPL | 15 |
| 発展1 比較演算子 | 10 |
| 発展2 IF / ELSE / THEN | 10 |


---
## 課題0: 第2回のコードを持ってくる

第2回の課題4 で書いたスタック操作を、ここに写しておくと楽になる。
この課題自体は採点しない。


In [ ]:
def stack_new():
    return []


def push(s, x):
    s.append(x)


def pop(s):
    if not s:
        raise IndexError('スタックが空です')
    return s.pop()


def peek(s):
    if not s:
        raise IndexError('スタックが空です')
    return s[-1]


In [ ]:
#@title 自動セットアップ (実行するだけ。中身は隠しています)
# [SKIP] 採点の対象外
# ---- 自己確認 (check / score) の準備。1回実行すればよい ----
import pathlib
import urllib.request

if not pathlib.Path('autograde.py').exists():
    try:
        urllib.request.urlretrieve(
            'https://3tty0n.github.io/2026-scripting-lab/grading/autograde.py', 'autograde.py')
    except Exception as e:
        print('取得できませんでした:', e)

from autograde import check, score
TASK = '03b'


---
## 課題1: 算術だけのインタプリタ (20点)

空白区切りの文字列を受け取り、スタックを書き換える `forth_eval(stack, text)` を書け。
対応するのは整数と `+ - * /` の4つだけでよい。

- 整数なら push
- 演算子なら2つ pop して計算し、結果を push
- 割り算は整数除算 `//` とする
- 知らないワードは `ValueError`

**ヒント**: `token.lstrip('-').isdigit()` で負数も含めて数値かどうか判定できる。
引き算・割り算は**順序に注意**: 先に pop されるのは右側の値。

### 例題

次のセルを実行して、動きを見る。

まず、整数と `+` だけを扱う版。これに `- * /` を足せば課題1 が終わる。


In [ ]:
# 例題
def tiny_eval(stack, text):
    for tok in text.split():
        if tok.lstrip('-').isdigit():
            push(stack, int(tok))          # 数はそのまま積む
        elif tok == '+':
            b = pop(stack)                   # 先に取れるのが右側
            a = pop(stack)
            push(stack, a + b)
        else:
            raise ValueError(f'知らないワード: {tok}')


s = stack_new()
tiny_eval(s, '3 4 +')
print(s)                                     # [7]


`3 4 -` は `3 - 4` で `-1`。`b` と `a` を逆にすると符号が狂う。
割り算は `a // b`。


In [ ]:
def forth_eval(stack, text):
    for token in text.split():
        if token == '+':
            b, a = pop(stack), pop(stack); push(stack, a + b)
        elif token == '-':
            b, a = pop(stack), pop(stack); push(stack, a - b)
        elif token == '*':
            b, a = pop(stack), pop(stack); push(stack, a * b)
        elif token == '/':
            b, a = pop(stack), pop(stack); push(stack, a // b)
        elif token.lstrip('-').isdigit():
            push(stack, int(token))
        else:
            raise ValueError(f'知らないワード: {token}')


def run(text):
    s = stack_new()
    forth_eval(s, text)
    return s


print(run('3 4 + 2 *'))

In [ ]:
# [CHECK] q1
need('forth_eval', arity=2)
need('run', arity=1)

assert run('3 4 +') == [7]
assert run('10 3 -') == [7]        # 順序に注意
assert run('3 4 + 2 *') == [14]
assert run('2 3 4 * +') == [14]
assert run('20 4 /') == [5]
assert run('1 2 3') == [1, 2, 3]
assert run('-5 3 +') == [-2]
assert run('') == []

# 性質で確かめる
holds('数を並べたらそのまま積まれる',
      lambda a, b: run(f'{a} {b}') == [a, b], ints(2, lo=0, hi=99))
holds('足してから引くと元の数',
      lambda a, b: run(f'{a} {b} + {b} -') == [a], ints(2, lo=0, hi=99))
try:
    run('3 FOO')
except ValueError:
    pass
else:
    raise AssertionError('知らないワードは ValueError を出すこと')
print('OK')


---
## 課題2: スタック操作ワード (20点)

`forth_eval` に次のワードを追加せよ。**大文字・小文字を区別しない**こと
(`DUP` も `dup` も動く)。

| ワード | 動作 | スタックの変化 |
|---|---|---|
| `.` | 一番上を**出力して**取り除く | (a → ) |
| `DUP` | 一番上を複製 | (a → a a) |
| `DROP` | 一番上を捨てる | (a → ) |
| `SWAP` | 上2つを入れ替え | (a b → b a) |
| `OVER` | 2番目を上にコピー | (a b → a b a) |
| `ROT` | 3番目を一番上へ | (a b c → b c a) |

**ヒント**: `if` を並べてもよいが、**`{ワード名: 関数}` の辞書**を作って
`BUILTINS[token](stack)` と呼ぶほうがずっと短い。この形にしておくと課題3 が楽になる。

### 例題

次のセルを実行して、動きを見る。

`if` の列を辞書に置き換える。ワードは「スタックを受け取る関数」1つずつ。


In [ ]:
# 例題
SAMPLE = {
    '+': binop(operator.add),
    'NEG': lambda s: push(s, -pop(s)),       # 課題には無い、形の見本
}


def forth_eval(stack, text):
    for tok in text.split():
        key = tok.upper()                  # 大文字小文字を区別しない
        if key in SAMPLE:
            SAMPLE[key](stack)
        elif tok.lstrip('-').isdigit():
            push(stack, int(tok))
        else:
            raise ValueError(f'知らないワード: {tok}')


`DUP` は「上を覗いて積む」、`SWAP` は「2つ取って逆順に積む」。
`.` は `print(pop(stack))`。
複数行が要るワードは `lambda` ではなく `def` で書き、あとから辞書に入れる。


In [ ]:
# 例題
def w_swap(s):
    b, a = pop(s), pop(s)
    push(s, b)
    push(s, a)


SAMPLE['SWAP'] = w_swap


In [ ]:
import operator


def binop(f):
    def run_(stack):
        b = pop(stack)
        a = pop(stack)
        push(stack, f(a, b))
    return run_


def _dot(s):
    print(pop(s))


def _swap(s):
    b, a = pop(s), pop(s); push(s, b); push(s, a)


def _over(s):
    b, a = pop(s), pop(s)
    for x in (a, b, a):
        push(s, x)


def _rot(s):
    c, b, a = pop(s), pop(s), pop(s)
    for x in (b, c, a):
        push(s, x)


BUILTINS = {
    '+': binop(operator.add), '-': binop(operator.sub),
    '*': binop(operator.mul), '/': binop(operator.floordiv),
    '.': _dot,
    'DUP': lambda s: push(s, peek(s)),
    'DROP': lambda s: pop(s),
    'SWAP': _swap, 'OVER': _over, 'ROT': _rot,
}


def forth_eval(stack, text):
    for token in text.split():
        up = token.upper()
        if up in BUILTINS:
            BUILTINS[up](stack)
        elif token.lstrip('-').isdigit():
            push(stack, int(token))
        else:
            raise ValueError(f'知らないワード: {token}')

In [ ]:
# [CHECK] q2
need('BUILTINS', kind=dict)
need('forth_eval', arity=2)

import io, contextlib

assert run('1 2 SWAP') == [2, 1]
assert run('1 2 swap') == [2, 1]     # 小文字でも動くこと
assert run('5 DUP') == [5, 5]
assert run('1 2 DROP') == [1]
assert run('1 2 OVER') == [1, 2, 1]
assert run('1 2 3 ROT') == [2, 3, 1]
assert run('5 dup *') == [25]
assert run('3 4 + 2 *') == [14]      # 課題1 が壊れていないこと
assert run('10 3 -') == [7]

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    s = run('42 .')
assert s == [], 'ドットは取り除くこと'
assert '42' in buf.getvalue(), 'ドットは出力すること'

try:
    run('FOO')
except ValueError:
    pass
else:
    raise AssertionError('知らないワードは ValueError を出すこと')
print('OK')


---
## 課題3: ユーザ定義ワード (25点)

`: 名前 本体 ;` で新しいワードを定義できるようにせよ。

```
: SQUARE DUP * ;
5 SQUARE .          →  25

: DOUBLE DUP + ;
: QUADRUPLE DOUBLE DOUBLE ;
5 QUADRUPLE .       →  20
```

定義は `USER_WORDS` に**トークンのリストとして**保存し、
その名前が出てきたら本体を評価する。
定義済みのワードを本体から呼べること (`QUADRUPLE` が `DOUBLE` を呼ぶ)。

**ヒント**: `:` に出会ったら、`;` が来るまでのトークンを集めてから辞書に入れる。
そのためにはトークンを**リストにして添字で進める**とよい。
本体の評価は `forth_eval` を**再帰的に呼ぶ**だけで済む。

### 例題

次のセルを実行して、動きを見る。

トークンを**添字で進める**形にすると、`;` まで読み飛ばせる。
下は `( ... )` をコメントとして飛ばす例。仕組みは `: ... ;` と同じ。


In [ ]:
# 例題
tokens = '1 ( これは注釈 ) 2 +'.split()
i = 0
while i < len(tokens):
    token = tokens[i]
    if token == '(':
        j = i + 1
        while tokens[j] != ')':              # 閉じるまで進める
            j += 1
        i = j                                # 末尾の i += 1 で ')' の次へ
    else:
        print('処理する:', token)
    i += 1


定義の場合は、`:` の次が名前、その後ろから `;` の手前までが本体。
本体は**リストのまま** `USER_WORDS[名前] = 本体` と入れる。

呼ばれたときは、その本体を評価するだけ。


In [ ]:
# 例題
if key in USER_WORDS:
    forth_eval(stack, USER_WORDS[key])       # 自分を呼ぶ (リストを渡せる)


`QUADRUPLE` が `DOUBLE` を呼べるのは、この再帰のおかげ。


In [ ]:
USER_WORDS = {}


def _find_else_then(tokens, start):
    """start から対応する ELSE / THEN の位置を返す"""
    depth, els = 0, None
    i = start
    while i < len(tokens):
        up = tokens[i].upper()
        if up == 'IF':
            depth += 1
        elif up == 'THEN':
            if depth == 0:
                return els, i
            depth -= 1
        elif up == 'ELSE' and depth == 0:
            els = i
        i += 1
    raise ValueError('対応する THEN がありません')


def forth_eval(stack, text):
    tokens = text.split() if isinstance(text, str) else text
    i = 0
    while i < len(tokens):
        token = tokens[i]
        up = token.upper()
        if token == ':':
            name = tokens[i + 1].upper()
            k = i + 2
            while tokens[k] != ';':
                k += 1
            USER_WORDS[name] = tokens[i + 2:k]
            i = k
        elif up == 'IF':
            cond = pop(stack)
            els, then = _find_else_then(tokens, i + 1)
            if cond != 0:
                forth_eval(stack, tokens[i + 1:els if els is not None else then])
            elif els is not None:
                forth_eval(stack, tokens[els + 1:then])
            i = then
        elif up in USER_WORDS:
            forth_eval(stack, USER_WORDS[up])
        elif up in BUILTINS:
            BUILTINS[up](stack)
        elif token.lstrip('-').isdigit():
            push(stack, int(token))
        else:
            raise ValueError(f'知らないワード: {token}')
        i += 1


USER_WORDS.clear()
print(run(': SQUARE DUP * ;  5 SQUARE'))

In [ ]:
# [CHECK] q3
need('USER_WORDS', kind=dict)
need('forth_eval', arity=2)

USER_WORDS.clear(); assert run(': SQUARE DUP * ; 5 SQUARE') == [25]
USER_WORDS.clear(); assert run(': DOUBLE DUP + ; : QUAD DOUBLE DOUBLE ; 5 QUAD') == [20]
USER_WORDS.clear(); assert run(': INC 1 + ; 10 INC INC INC') == [13]
USER_WORDS.clear(); assert run('1 : ADD2 2 + ; ADD2 ADD2') == [5]
USER_WORDS.clear(); assert run(': SQ DUP * ; 3 SQ SQ') == [81]

# 定義しただけでは何もスタックに積まれない
USER_WORDS.clear(); assert run(': NOP ;') == []

# 大文字小文字を区別しないこと
USER_WORDS.clear(); assert run(': SQUARE DUP * ; 4 square') == [16]

# 組み込みが壊れていないこと
USER_WORDS.clear()
assert run('3 4 + 2 *') == [14]
assert run('1 2 3 ROT') == [2, 3, 1]
# ユーザ定義は USER_WORDS に入れ、組み込みの表は汚さないこと
USER_WORDS.clear()
_before = set(BUILTINS)
run(': SQUARE DUP * ;')
assert set(BUILTINS) == _before, 'ユーザ定義を BUILTINS に混ぜないこと'
assert len(USER_WORDS) == 1, f'USER_WORDS が {len(USER_WORDS)} 件'
note('ユーザ定義ワードは組み込みと別の表に入っている')

print('OK')


---
## 課題4: 対話ループ (REPL) (15点)

1行ずつ評価して、結果を返すループを書け。
**採点できるように、`input()` ではなく引数で入力を受け取る形にする。**

`repl(lines)` を書け。`lines` は入力行のリスト。
返り値は**各行に対する出力文字列のリスト**。

| 入力行 | 返すもの |
|---|---|
| 普通の行 | 評価後のスタックを `str()` にしたもの (例: `'[7]'`) |
| `bye` | そこで終了。それ以降の行は処理しない |
| エラーが起きた行 | `'error: ' + エラーメッセージ` を返し、**ループは続ける** |

スタックは行をまたいで保たれること (1行目で積んだ値を2行目で使える)。

**ヒント**: 第2回の例外処理。`except Exception as e:` で捕まえて `str(e)` を使う。

### 例題

次のセルを実行して、動きを見る。

1行ずつ、失敗しても続ける形。


In [ ]:
# 例題
outputs = []
stack = stack_new()
for line in ['3 4 +', 'FOO']:
    try:
        forth_eval(stack, line)
        outputs.append(str(stack))           # '[7]' のような文字列
    except Exception as e:
        outputs.append('error: ' + str(e))
print(outputs)                               # ['[7]', 'error: 知らないワード: FOO']


`bye` は `try` に入る前に判定して `break`。
`stack` をループの外で作っているので、値は行をまたいで残る。


In [ ]:
def repl(lines):
    stack = stack_new()
    outputs = []
    for line in lines:
        if line.strip().lower() == 'bye':
            break
        try:
            forth_eval(stack, line)
            outputs.append(str(stack))
        except Exception as e:
            outputs.append('error: ' + str(e))
    return outputs


USER_WORDS.clear()
for out in repl(['3 4 +', '2 *', 'FOO', ': SQ DUP * ;', 'SQ', 'bye', '999']):
    print(out)

In [ ]:
# 対話用 (採点しない)。実行すると入力待ち
def repl_interactive():
    stack = stack_new()
    while True:
        line = input('> ')
        if line.strip().lower() == 'bye':
            break
        try:
            forth_eval(stack, line)
            print(stack)
        except Exception as e:
            print('error:', e)


# repl_interactive()   ← コメントを外すと対話できる


In [ ]:
# [CHECK] q4
need('repl', arity=1)

USER_WORDS.clear()
out = repl(['3 4 +', '2 *'])
assert out == ['[7]', '[14]'], out

# bye で止まり、それ以降は処理しない
USER_WORDS.clear()
out = repl(['1 2 +', 'bye', '999'])
assert out == ['[3]'], out

# エラーでもループは続く
USER_WORDS.clear()
out = repl(['1 2 +', 'FOO', '10 +'])
assert len(out) == 3, out
assert out[0] == '[3]'
assert out[1].startswith('error:'), out[1]
assert out[2] == '[13]', 'エラーのあともスタックが保たれること'

# ユーザ定義ワードも行をまたいで残る
USER_WORDS.clear()
out = repl([': SQ DUP * ;', '5 SQ'])
assert out == ['[]', '[25]'], out

assert repl([]) == []
assert repl(['bye']) == []
print('OK')


---
# 発展課題

余力があれば。


## 発展1: 比較演算子 (10点)

`<` `>` `=` を追加せよ。Forth では**真が `-1`、偽が `0`** で表される
(歴史的な理由: すべてのビットが 1)。

### 例題

次のセルを実行して、動きを見る。

比較の結果を Forth の真偽値に直す。


In [ ]:
# 例題
BUILTINS['<'] = binop(lambda a, b: -1 if a < b else 0)


`>` と `=` も同じ形。Python の `True` を積むと `1` になってしまうので、
`-1` / `0` に変換すること。


In [ ]:
def _flag(f):
    return binop(lambda a, b: -1 if f(a, b) else 0)


BUILTINS['<'] = _flag(operator.lt)
BUILTINS['>'] = _flag(operator.gt)
BUILTINS['='] = _flag(operator.eq)

In [ ]:
# [CHECK] b1
USER_WORDS.clear()
assert run('3 4 <') == [-1]
assert run('4 3 <') == [0]
assert run('3 4 >') == [0]
assert run('4 3 >') == [-1]
assert run('4 4 =') == [-1]
assert run('3 4 =') == [0]
assert run('4 4 <') == [0] and run('4 4 >') == [0]
assert run('1 2 + 3 =') == [-1]
print('OK')


## 発展2: 条件分岐 (10点)

`IF ... ELSE ... THEN` を実装せよ。`IF` の直前にスタックにある値が
0 以外なら `IF` と `ELSE` の間を、0 なら `ELSE` と `THEN` の間を実行する
(`ELSE` は省略できる)。

```
: ABS DUP 0 < IF 0 SWAP - THEN ;
-5 ABS .     →  5
```

**ヒント**: 課題3 と同じく、トークン列の上で対応する `ELSE` / `THEN` を探す。
`IF` が入れ子になりうることに注意 (深さを数える)。

### 例題

次のセルを実行して、動きを見る。

入れ子を数えながら、対応する `THEN` を探す。


In [ ]:
# 例題
def find_then(tokens, i):
    """tokens[i] == 'IF' のとき、対応する THEN の位置を返す"""
    depth = 1
    j = i + 1
    while j < len(tokens):
        if tokens[j].upper() == 'IF':
            depth += 1                       # 内側の IF
        elif tokens[j].upper() == 'THEN':
            depth -= 1
            if depth == 0:
                return j
        j += 1
    raise ValueError('THEN がない')


`ELSE` も同じ要領で、`depth == 1` のものだけを自分のものとして拾う。
条件は `IF` に来た時点で `pop(stack)`。実行する側のトークン列を切り出して
`forth_eval(stack, 切り出した部分)` に渡し、`i` は `THEN` の位置まで飛ばす。


In [ ]:
# IF / ELSE / THEN は上の forth_eval に組み込み済み
print(run('1 IF 10 ELSE 20 THEN'), run('0 IF 10 ELSE 20 THEN'))

In [ ]:
# [CHECK] b2
USER_WORDS.clear()
assert run('1 IF 42 THEN') == [42]
assert run('0 IF 42 THEN') == []
assert run('1 IF 10 ELSE 20 THEN') == [10]
assert run('0 IF 10 ELSE 20 THEN') == [20]
assert run('-1 IF 7 THEN') == [7], '0 以外はすべて真'

USER_WORDS.clear()
assert run(': ABS DUP 0 < IF 0 SWAP - THEN ; -5 ABS') == [5]
USER_WORDS.clear()
assert run(': ABS DUP 0 < IF 0 SWAP - THEN ; 5 ABS') == [5]

# 入れ子
USER_WORDS.clear()
assert run('1 IF 1 IF 7 ELSE 8 THEN ELSE 9 THEN') == [7]
assert run('1 IF 0 IF 7 ELSE 8 THEN ELSE 9 THEN') == [8]
assert run('0 IF 1 IF 7 ELSE 8 THEN ELSE 9 THEN') == [9]

# 条件分岐の外側は普通に続く
assert run('1 IF 10 THEN 5 +') == [15]
print('OK')


---
## 自己確認

`score()` で全部まとめて確かめる。何度やってもよい。
`NG` の行に、次に何をすればいいかが出る。


In [ ]:
# [SKIP] 採点の対象外
score()


---
## 提出

提出は kibaco から。`score()` と採点ページは下書き確認用。

1. 上から実行し、出力を残す
2. `ファイル → .ipynb をダウンロード`
3. 締切までに kibaco へ出す

確認セルが全部 `OK` なら提出してよい。確認セルを消しても教員の採点は別の検査で行う。
